# Joint translation effect — liquid water

Compare cross sections for the reference and samples at 295 K. Translation sampling reuses the normalized vibrational spectrum.

## 0. Choose the engine

**Required setting**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `ENGINE` | `"ncrystal"` | Select `"njoy"` or `"ncrystal"` for this run. |

After changing the engine, restart the kernel and run all cells.

In [ ]:
AVAILABLE_ENGINE = ("njoy", "ncrystal")
AVAILABLE_ENGINE_LABEL = {
    "njoy": "NJOY",
    "ncrystal": "NCrystal"
}

In [ ]:
ENGINE = "ncrystal"
if ENGINE not in AVAILABLE_ENGINE:
    raise ValueError('Choose ENGINE = "njoy" or "ncrystal".')
ENGINE_LABEL = AVAILABLE_ENGINE_LABEL[ENGINE]

## 1. Locate TARA

The next cell finds TARA and imports the APIs for the chosen engine.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "core" / "config.py").is_file()
     and (path / "processing" / "pipelines" / "ace_pipeline.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the TARA repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import display

from core.config import TaraConfig
from core.data.input_pdos import PhononSpectra
from core.sampler.concrete_samplers import TaraSampler

if ENGINE == "njoy":
    from input_output.leapr.core import LeaprFile
    from input_output.leapr.material_adapter import material_model_from_leapr
elif ENGINE == "ncrystal":
    import NCrystal as NC
    from scipy.constants import physical_constants

    from core.data.material_scattering import (
        MaterialScatteringModel,
        ScatteringSpecies,
        SpeciesDynamics,
    )
    from input_output.ncrystal.core import NCrystalFile
    from input_output.ncrystal.liquids import load_liquid_ncmat, parse_custom_liquids

## 2. Select the required inputs

| Setting | Example value | What it controls |
| --- | --- | --- |
| `BASE_LEAPR_FILE` | `tests/fixtures/tsl-HinH2O.leapr` | NJOY material template. |
| `BASE_NCMAT_FILE` | `tests/fixtures/liquid_water_295K_illustrative.ncmat` | NCrystal material. |
| `BASE_ENDF_FILE` | `tests/fixtures/n_1-H-001g.jeff` | Neutron evaluation for the NJOY path. Override with `TARA_BASE_ENDF` if needed. |
| `NJOY_EXECUTABLE` | `NJOY_EXEC` or `njoy` on `PATH` | Executable for the NJOY path. |

Edit paths only if your checkout or NJOY installation uses different locations.

NCrystal liquid calculations require the Liquids plugin.

In [ ]:
fixtures = PROJECT_ROOT / "tests" / "fixtures"
BASE_ENDF_FILE = Path(os.environ.get("TARA_BASE_ENDF", str(fixtures / "n_1-H-001g.jeff")))
njoy_path = shutil.which(os.environ.get("NJOY_EXEC") or "njoy")
NJOY_EXECUTABLE = Path(njoy_path) if njoy_path else None

if ENGINE == "njoy":
    if not BASE_ENDF_FILE.is_file() or BASE_ENDF_FILE.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty neutron ENDF: {BASE_ENDF_FILE}")
    if NJOY_EXECUTABLE is None:
        raise FileNotFoundError("Set NJOY_EXEC or add njoy to PATH.")

    print("Neutron ENDF:", BASE_ENDF_FILE)
    print("NJOY:", NJOY_EXECUTABLE)

In [ ]:
if ENGINE == "njoy":
    BASE_LEAPR_FILE = fixtures / "tsl-HinH2O.leapr"
    if not BASE_LEAPR_FILE.is_file():
        raise FileNotFoundError(f"LEAPR template missing: {BASE_LEAPR_FILE}")
elif ENGINE == "ncrystal":
    BASE_NCMAT_FILE = fixtures / "liquid_water_295K_illustrative.ncmat"
    if not BASE_NCMAT_FILE.is_file():
        raise FileNotFoundError(f"NCMAT template missing: {BASE_NCMAT_FILE}")

## 3. Load the liquid water reference

**Fixed settings for this example**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `TEMPERATURE_K` | `295` | Evaluation temperature in kelvin. |
| `ATOM_LABEL` | `"H"` | Species supplying the reference spectrum. |

Both engines load a physical material model with the continuous spectrum, component weights, and Egelstaff–Schofield parameter used for sampling.

In [ ]:
TEMPERATURE_K = 295.0
ATOM_LABEL = "H"
print(f"Engine: {ENGINE_LABEL}")
print(f"Temperature: {TEMPERATURE_K:g} K")

if ENGINE == "njoy":
    template = LeaprFile.from_file(str(BASE_LEAPR_FILE))
    material = material_model_from_leapr(
        template,
        state="liquid",
        temperature=TEMPERATURE_K,
        principal_label=ATOM_LABEL,
        secondary_label="O",
    )
    if (material.principal.multiplicity != 2 or len(material.secondary) != 1
            or material.secondary[0].multiplicity != 1):
        raise ValueError("Expected the CAB H-in-water LEAPR material.")
    print(f"LEAPR file: {BASE_LEAPR_FILE}")
elif ENGINE == "ncrystal":
    source_text = load_liquid_ncmat(BASE_NCMAT_FILE)
    liquid_entries = parse_custom_liquids(source_text)
    liquid_settings = next((entry for entry in liquid_entries if entry.atom == ATOM_LABEL), None)
    if liquid_settings is None:
        raise ValueError(f"No custom liquid settings for {ATOM_LABEL!r}.")

    # Read the spectrum from the NCMAT file and build the physical sampling input.
    ncmat = NCrystalFile.from_file(str(BASE_NCMAT_FILE))
    spectrum = PhononSpectra(pdos=ncmat.pdos_for_atom(ATOM_LABEL))
    neutron_mass = physical_constants["neutron mass in u"][0]
    hydrogen_mass = NC.atomDB(ATOM_LABEL).averageMassAMU() / neutron_mass
    oxygen_mass = NC.atomDB("O").averageMassAMU() / neutron_mass
    material = MaterialScatteringModel(
        identifier=BASE_NCMAT_FILE.stem,
        state="liquid",
        temperature_k=TEMPERATURE_K,
        principal=ScatteringSpecies(ATOM_LABEL, 2, hydrogen_mass),
        dynamics=SpeciesDynamics(
            continuous=spectrum,
            continuous_weight=liquid_settings.ws,
            translation_weight=liquid_settings.wt,
            translation_model="egelstaff_schofield",
            translation_parameter=liquid_settings.c,
        ),
        secondary=(ScatteringSpecies("O", 1, oxygen_mass),),
    )
    print(f"NCMAT file: {BASE_NCMAT_FILE}")

dynamics = material.dynamics
if dynamics.translation_model != "egelstaff_schofield":
    raise ValueError("Select a reference with Egelstaff–Schofield translation.")
vibrational_weight = dynamics.continuous_weight + sum(
    oscillator.weight for oscillator in dynamics.oscillators
)
scaled_diffusivity = dynamics.translation_weight * dynamics.translation_parameter
print(f"Vibrational weight: {vibrational_weight:g}")
print(f"Translation weight: {dynamics.translation_weight:g}")
print(f"Reference diffusion parameter (c): {dynamics.translation_parameter:.7g}")
print(f"Reference scaled diffusivity: {scaled_diffusivity:.7g}")

## 4. Set the translation sampling controls

These bounds define a sensitivity study. They are relative to the reference values.

**Required settings**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `MAX_WEIGHT_SHIFT_PCT` | `10.0` | Maximum relative change in translation weight, as a percentage. |
| `MAX_DIFFUSIVITY_SHIFT_PCT` | `10.0` | Maximum relative change in scaled diffusivity, as a percentage. |
| `N_SAMPLES` | `3` | Stochastic realizations in addition to the reference. |

**Optional settings**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `RANDOM_SEED` | `42` | Reproduces the Latin hypercube draws. |

In [ ]:
MAX_WEIGHT_SHIFT_PCT = 10.0
MAX_DIFFUSIVITY_SHIFT_PCT = 10.0
N_SAMPLES = 3
RANDOM_SEED = 42

In [ ]:
# Set the translation mode and its perturbation bounds.
diffusion_configuration = dict(
    enabled=True,
    mode="joint_independent",
    max_weight_shift_pct=MAX_WEIGHT_SHIFT_PCT,
    max_diffusivity_shift_pct=MAX_DIFFUSIVITY_SHIFT_PCT,
    weight_compensation="continuous_only",
)

# Validate the perturbation bounds and sampling settings.
configuration = TaraConfig(
    n_samples=N_SAMPLES,
    random_seed=RANDOM_SEED,
    diffusion=diffusion_configuration,
)

# Initialize the sampler with the physical reference material.
sampler = TaraSampler(
    config=configuration,
    material=material,
)

# Generate the translation ensemble.
ensemble = sampler.generate_ensemble()
if ensemble.reference is None or len(ensemble) != N_SAMPLES:
    raise RuntimeError("Expected the reference and all requested samples.")
print(f"Prepared one reference and {len(ensemble)} stochastic samples.")

## 5. Check the translation sampling results

In [ ]:
info = ensemble.sample_info.copy()
info.index = [f"sample_{index + 1:03d}" for index in range(len(ensemble))]
info.index.name = "Sample"
print("sample_000 is the reference. Factors below are relative to that material.")

if ENGINE == "njoy":
    translation_label = "twt"
elif ENGINE == "ncrystal":
    translation_label = "wt"

scaled_label = f"{translation_label} × c"
column_labels = {
    "translation_weight": translation_label,
    "diffusion_parameter_c": "c",
    "scaled_diffusion_d": scaled_label,
    "diffusivity_factor": "Diffusivity factor",
    "weight_factor": "Weight factor",
}
display(info[list(column_labels)].rename(columns=column_labels))

## 6. Configure the Analyzer and request cross sections

Use the material ensemble generated in Section 4. The runner writes each sample's translation settings before evaluating its cross sections. Each sample uses the reference vibrational shape.

- **NJOY:** LEAPR and THERMR evaluate the CAB hydrogen thermal scattering channels. The total excludes absorption. This example has no separate elastic channel.
- **NCrystal:** Native scattering includes hydrogen and the free-gas oxygen contribution, in barns per atom. Absorption is a separate optional calculation.

| Setting | Example value | What it controls |
| --- | --- | --- |
| `EVAL_ENERGY_GRID` | `np.geomspace(1e-5, 1.0, 180)` | Incident neutron energies in eV. NCrystal's runner and parser share this grid. |
| `calc_xs_total`, `calc_xs_inelastic` | `True` | Cross-section channels requested from the parser. |

In [ ]:
import numpy as np

from analyzer.ensemble_analyzer import EnsembleAnalyzer

if ENGINE == "njoy":
    from scipy.constants import physical_constants
    from analyzer.parsers.njoy_parser import NjoyParser
    from analyzer.runners.njoy_runner import NjoyRunner

    mass_amu = material.principal.mass_ratio * physical_constants["neutron mass in u"][0]
    runner = NjoyRunner(
        njoy_exec=str(NJOY_EXECUTABLE),
        base_endf_file=str(BASE_ENDF_FILE),
        base_leapr_file=str(BASE_LEAPR_FILE),
        temperature=TEMPERATURE_K,
        mass_amu=mass_amu,
    )
    parser = NjoyParser()
else:
    from analyzer.parsers.ncrystal_parser import NCrystalParser
    from analyzer.runners.ncrystal_runner import NCrystalRunner

    runner = NCrystalRunner(
        base_ncmat_file=str(BASE_NCMAT_FILE),
        temperature=TEMPERATURE_K
    )
    parser = NCrystalParser()

In [ ]:
# Initialize the ensemble analyzer with the generated ensemble, runner, and parser.
analyzer = EnsembleAnalyzer(
    ensemble=ensemble,
    runner=runner,
    parser=parser
)

In [ ]:
EVAL_ENERGY_GRID = np.geomspace(1e-5, 1.0, 180)

parser_kwargs = {
    "energy_grid": EVAL_ENERGY_GRID,
    "calc_xs_total": True,
    "calc_xs_inelastic": True,
    "calc_xs_elastic": False,
    "calc_thermodynamics": False,
}
if ENGINE == "ncrystal":
    parser_kwargs["calc_xs_absorption"] = False

## 7. Run the Analyzer

Evaluate `sample_000` separately as the reference. The Analyzer keeps the remaining samples aligned with the translation parameter table in Section 5.

| Setting | Example value | What it controls |
| --- | --- | --- |
| `RUN_PARALLEL` | `False` | Run sequentially or request concurrent processing. |
| `thermr_emax` | `1.0` | NJOY THERMR upper energy in eV. |
| `thermr_tol` | `0.01` | NJOY THERMR interpolation tolerance. |
| `thermr_nbin` | `16` | NJOY THERMR angular bins. |

In [ ]:
RUN_PARALLEL = False

if ENGINE == "njoy":
    RUNNER_OPTIONS = {
        "thermr_emax": 1.0,
        "thermr_tol": 0.01,
        "thermr_nbin": 16
    }
else:
    RUNNER_OPTIONS = {
        "atom_label": ATOM_LABEL,
        "energy_grid": EVAL_ENERGY_GRID
    }

In [ ]:
# Compute the ensemble results using the configured analyzer.
analyzer.compute(
    parallel=RUN_PARALLEL,
    runner_kwargs=RUNNER_OPTIONS,
    parser_kwargs=parser_kwargs,
    compute_reference=True,
)

if analyzer.reference_result is None or len(analyzer.raw_results) != len(ensemble):
    raise RuntimeError("Expected one reference and one result per stochastic material.")

print(f"Analyzed one reference and {len(analyzer.raw_results)} stochastic materials.")

## 8. Plot the cross sections

Each dashboard shows the three vibrational regions in meV, the selected cross section against incident energy in eV, and the percentage difference from the independently evaluated reference, $100(\sigma_{\mathrm{sample}}-\sigma_{\mathrm{ref}})/\sigma_{\mathrm{ref}}$. Relative differences are undefined where the reference is zero.

Use `LOW_ENERGY_MAX_EV` to focus the two cross-section rows on low incident energies. Set it to `None` to show the full range. This setting changes the plot limits only.

In [ ]:
import matplotlib.pyplot as plt

from notebooks.tara_plotting import plot_water_master_dashboard

LOW_ENERGY_MAX_EV = None


def show_cross_section_channel(channel, label):
    """Show the requested channel with the shared water dashboard.

    Parameters
    ----------
    channel : str
        Analyzer cross-section field.
    label : str
        Title for the selected channel.

    Returns
    -------
    None
        Displays the dashboard when the channel is available.
    """
    results = [analyzer.reference_result, *analyzer.raw_results]
    available = [getattr(result.integ, channel) is not None for result in results]
    if not available[0] or not any(available[1:]):
        print(f"{label}: unavailable for this calculation.")
        return

    _, axes = plot_water_master_dashboard(
        data_obj=analyzer,
        ref_label=f"{ENGINE_LABEL} reference",
        main_title=label,
        xs_type=channel,
        plot_pdos=True,
        plot_xs=True,
        plot_error=True,
    )
    if LOW_ENERGY_MAX_EV is not None:
        axes["xs"].set_xlim(EVAL_ENERGY_GRID[0], LOW_ENERGY_MAX_EV)
    plt.show()

### 8.1. Total scattering cross section

In [ ]:
total_label = ("Total thermal scattering cross section" if ENGINE == "njoy"
               else "Total scattering cross section")
show_cross_section_channel("xs_total", total_label)

### 8.2. Inelastic scattering cross section

In [ ]:
show_cross_section_channel("xs_inelastic", "Inelastic scattering cross section")

Compare both factors in Step 5 with the low-energy differences. The two companion notebooks isolate each effect separately.

See also [Diffusivity at fixed translation weight](01_diffusivity_effect.ipynb) and [Translation weight at fixed diffusivity](02_translation_weight_effect.ipynb).

[Cross-section analysis](../02_analysis/cross_sections/liquid.ipynb) · [Liquid ACE generation](../03_ace_generation/liquid.ipynb)